# Exploring a topology-broad trajectory representation space

This notebook tests the first milestone from the active-sampling research plan: represent smooth, physically band-limited paths without restricting them to zigzags. It compares human-readable primitives, unconstrained smooth paths, and protected reverse traversals before introducing a neural operator.

The path geometry is a variable-complexity cubic B-spline. Normal force and speed are continuous channels along arc-length parameter $s$. The plots use normalized sheet coordinates $[0,1]^2$; color denotes force, the green marker is the start, and the red marker is the end.

**Environment:** NumPy, SciPy, and Matplotlib are required. All cells were validated with the project's `smcpp_vtk38` Python environment. The same figures can be regenerated without Jupyter by running `python/path_representation_space.py` in that environment.

In [ ]:
from collections import Counter
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'python'))

from path_representation_space import (
    DESCRIPTOR_NAMES, build_candidate_pool, compare_initial_designs,
    compare_representation_designs, descriptor_matrix,
    evaluate_flip_invariance, evaluate_trajectory_metrics, pca_embedding,
    plot_collision_pairs, plot_descriptor_coverage, plot_design_metrics,
    plot_embedding, plot_flip_invariance_results, plot_gallery,
    plot_metric_learning_results,
    plot_metric_response_alignment, source_counts, standardize_descriptors,
)

seed = 7
output_dir = repo_root / 'notebooks' / 'path_representation_outputs'
output_dir.mkdir(parents=True, exist_ok=True)
plt.rcParams['figure.dpi'] = 115

## 1. Generate a mixed candidate pool

The candidate generator does not assume that a useful path belongs to a human path family. Lines, arcs, spirals, and zigzags are retained as interpretable seeds; most candidates are free-form smooth draws. A protected set is traversed in reverse so direction-dependent descriptors and later order effects are not left to chance.

In [ ]:
paths = build_candidate_pool(seed=seed)
print(f'{len(paths)} feasible paths')
print(dict(source_counts(paths)))

rng = np.random.default_rng(seed + 1)
gallery_indices = []
for source in sorted(set(path.source for path in paths)):
    members = [i for i, path in enumerate(paths) if path.source == source]
    gallery_indices.extend(rng.choice(members, size=min(3, len(members)), replace=False))

fig = plot_gallery(
    paths, gallery_indices,
    'Representative candidates from a topology-broad path space',
    output=output_dir / 'candidate_gallery.png',
)
plt.show()

## 2. Describe paths in physically meaningful coordinates

Raw spline coefficients are a poor coverage metric: two coefficient vectors can describe almost the same path, and small coefficient changes can have different effects at different complexities. We instead compute descriptors for length, duration, curvature, turning, swept area, revisit fraction, location, boundary clearance, closure, directional endpoints, process channels, dose, and complexity.

In [ ]:
descriptors = descriptor_matrix(paths)
standardized, descriptor_center, descriptor_scale = standardize_descriptors(descriptors)
embedding, explained = pca_embedding(standardized)

print(f'descriptor matrix: {descriptors.shape}')
print(f'first two PCA axes explain {100 * explained.sum():.1f}% of robust-standardized variance')
print(', '.join(DESCRIPTOR_NAMES))

## 3. Select an initial design by descriptor-space maximin coverage

Greedy farthest-point selection is only a transparent baseline, but it answers the immediate question: can a small initial batch cover a much broader representation space than a zigzag-only or random design? The covering radius is measured over the complete candidate pool, so lower is better.

In [ ]:
comparison = compare_initial_designs(
    paths, standardized, batch_size=24, n_random_trials=100, seed=seed + 2
)
selected = comparison['mixed_indices']
random_radii = comparison['random_radii']

print(f"mixed maximin radius : {comparison['mixed_radius']:.3f}")
print(f"zigzag-only radius   : {comparison['zigzag_radius']:.3f}")
print(f'random radius         : {random_radii.mean():.3f} +/- {random_radii.std():.3f}')
print('selected sources      :', dict(Counter(paths[i].source for i in selected)))

fig = plot_embedding(
    paths, embedding, selected, explained,
    output=output_dir / 'descriptor_embedding.png',
)
plt.show()

In [ ]:
fig = plot_gallery(
    paths, selected, 'Descriptor-maximin initial design',
    output=output_dir / 'selected_initial_design.png',
)
plt.show()

fig = plot_descriptor_coverage(
    descriptors, selected, output=output_dir / 'descriptor_coverage.png'
)
plt.show()

## 4. Inspect a protected forward/reverse pair

A reverse path occupies the same spatial curve, so a purely spatial representation would collapse the two. Directional start/end descriptors and ordered force/speed channels keep them distinct. The physical response will only become distinguishable once a history-dependent oracle is added.

In [ ]:
reverse_index = next(i for i, path in enumerate(paths) if path.source == 'reverse')
forward_label = str(paths[reverse_index].label)[len('reverse_'):]
forward_index = next(i for i, path in enumerate(paths) if path.label == forward_label)

fig = plot_gallery(
    paths, [forward_index, reverse_index],
    'Same spatial curve, different traversal history',
    output=output_dir / 'forward_reverse_pair.png', n_columns=2,
)
plt.show()

difference = descriptors[forward_index] - descriptors[reverse_index]
changed = [(name, value) for name, value in zip(DESCRIPTOR_NAMES, difference) if abs(value) > 1e-10]
changed

## 5. Compare diversity in the actual ordered surrogate input

Descriptor diversity is not a guarantee of diversity in the sequence consumed by a surrogate. We therefore canonicalize every path by normalized cumulative arc length and encode the ordered geometry, tangent, force, speed, length, duration, and dose. The comparison below selects 24 paths using six policies and evaluates each selection in four different ways. Covering radius and MMD are better when lower; effective rank is better when higher.

In [ ]:
representation = compare_representation_designs(paths, batch_size=24, seed=seed + 14)
hybrid_selected = representation['selections']['hybrid kernel D-opt']
input_embedding, input_explained = pca_embedding(representation['input_normalized'])

for name, metrics in representation['metrics'].items():
    print(
        f"{name:22s} d_desc={metrics['descriptor_radius']:.3f}  "
        f"d_input={metrics['input_radius']:.3f}  MMD={metrics['input_mmd']:.3f}  "
        f"rank={metrics['kernel_effective_rank']:.2f}"
    )

fig = plot_embedding(
    paths, input_embedding, hybrid_selected, input_explained,
    output=output_dir / 'ordered_input_embedding.png',
    space_label='canonical ordered input',
)
plt.show()

fig = plot_design_metrics(
    representation['metrics'],
    output=output_dir / 'representation_design_comparison.png',
)
plt.show()

fig = plot_gallery(
    paths, hybrid_selected, 'Hybrid input/descriptor kernel D-optimal design',
    output=output_dir / 'hybrid_initial_design.png',
)
plt.show()

## 6. Audit descriptor/input representation collisions

The collision search finds pairs in the closest 10% under one metric that are maximally separated under the other. These pairs demonstrate directly why no single handcrafted descriptor map can be assumed to preserve the surrogate's ordered input geometry.

In [ ]:
for kind, pairs in representation['collisions'].items():
    print(f'\n{kind}:')
    for i, j, d_desc, d_input in pairs:
        print(
            f'  {paths[i].label:18s} vs {paths[j].label:18s} '
            f'd_desc={d_desc:.2f}, d_input={d_input:.2f}'
        )

fig = plot_collision_pairs(
    paths, representation['collisions'],
    output=output_dir / 'representation_collisions.png',
)
plt.show()

## 7. Audit left/right and up/down flip invariance

For a centered homogeneous rectangle, the in-plane symmetry group is $D_2$: identity, horizontal reflection, vertical reflection, and 180-degree rotation. A physical reflection transforms every time-indexed path point but does not reverse time. We test two distinct properties: simultaneous isometry, $d(gT_1,gT_2)=d(T_1,T_2)$, and quotient invariance, $d_{D_2}(T,gT)=0$ with $d_{D_2}(T_1,T_2)=\min_h d(T_1,hT_2)$.

In [ ]:
flip_audit = evaluate_flip_invariance(paths, seed=53)
for name, metrics in flip_audit['metrics'].items():
    print(
        f"{name:18s} isometry_max={metrics['simultaneous_isometry_max_error']:.2e}  "
        f"raw_orbit={metrics['raw_orbit_distance_median']:.3f}  "
        f"quotient_max={metrics['quotient_orbit_distance_max']:.2e}"
    )

fig = plot_flip_invariance_results(
    flip_audit['metrics'],
    output=output_dir / 'trajectory_flip_invariance.png',
)
plt.show()

orbit = flip_audit['example_orbit']
fig = plot_gallery(
    orbit, range(len(orbit)), 'One trajectory orbit under rectangle D2 symmetry',
    output=output_dir / 'trajectory_flip_orbit.png', n_columns=4,
)
plt.show()

### Symmetry conclusion

All seven original metrics are simultaneous $D_2$ isometries to numerical precision, but their raw distances correctly remain nonzero between a path and a spatially flipped copy. Flip invariance is obtained by taking the minimum over the symmetry orbit. Therefore the simplest starting point is the $D_2$-quotient canonical ordered H1 metric: it is inexpensive, differentiable, order/process aware, and does not require DTW, Fréchet, or a kernel. More complex quotient metrics remain comparison baselines.

## 8. Test which metric supports learning on a history-dependent oracle

Trajectory clustering quality is not learning utility. This controlled benchmark applies each path to a spatial state with localized deposition, exponential hardening, overlap, and order-dependent directional feedback. It is deliberately inexpensive and is **not** claimed to be an English-wheel model. The response contains spatial strain/hardening fields and global moments.

Seven distances are compared: canonical ordered H1, descriptors, weighted discrete Fréchet, constrained DTW, sliced occupation Wasserstein, a level-two path signature, and an equal-weight multi-view combination. Each metric is judged by response-distance rank correlation, response-neighborhood preservation, response-triplet ordering, response-space radius of an 18-path maximin design, held-out response kNN error, Gaussian-kernel definiteness, and pairwise runtime.

In [ ]:
metric_benchmark = evaluate_trajectory_metrics(paths, batch_size=18, seed=31)
print(f"benchmark paths: {len(metric_benchmark['paths'])}")
print(f"active response features: {metric_benchmark['active_response_features'].sum()}")
for name, metrics in sorted(
    metric_benchmark['metrics'].items(), key=lambda item: item[1]['knn_error']
):
    print(
        f"{name:18s} rho={metrics['response_spearman']:.3f}  "
        f"NN={metrics['neighbor_overlap']:.3f}  triplet={metrics['triplet_agreement']:.3f}  "
        f"radius={metrics['response_radius']:.3f}  error={metrics['knn_error']:.3f}  "
        f"min_eig={metrics['kernel_min_eigenvalue']:.3f}  "
        f"time={metrics['runtime_seconds']:.3f}s"
    )

fig = plot_metric_learning_results(
    metric_benchmark['metrics'],
    output=output_dir / 'trajectory_metric_learning_results.png',
)
plt.show()

fig = plot_metric_response_alignment(
    metric_benchmark,
    output=output_dir / 'trajectory_metric_response_alignment.png',
)
plt.show()

### Interpretation

For this oracle, the multi-view distance gives the best combined response rank correlation, triplet agreement, response-space coverage, and held-out kNN error. Constrained DTW and occupation Wasserstein preserve response neighborhoods well. Canonical H1 and the truncated signature remain useful inexpensive positive-definite-kernel representations. Naïve Gaussian kernels built from Fréchet, DTW, occupation Wasserstein, and the multi-view distance are indefinite here, so those distances cannot be inserted blindly into kernel D-optimality.

This ranking is task dependent: changing the oracle's memory, noise, or rate sensitivity can change the winner. The experiment demonstrates the evaluation protocol, not a universal best trajectory metric.

## 9. What this prototype can and cannot establish

This notebook establishes that broad smooth paths can be represented and selected in multiple non-equivalent geometries, and that a metric can be evaluated by how well it supports learning a known history-dependent response. Input maximin and input-kernel D-optimality cover ordered-input extremes, random sampling best matches the candidate distribution by MMD, descriptor maximin covers handcrafted summaries, and the hybrid balances them. The collision audit proves that descriptor and ordered-input neighborhoods do not carry over automatically.

The synthetic oracle now provides an initial learning test, but it does **not** determine the best metric for the shell simulator. The next step is to expose the same metric API to repository simulations and evaluate each design by held-out shell-response error at equal solver cost. Until then, multi-view distance is the best candidate for this controlled oracle, not a proven physical optimum.